In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("dirty_ecommerce_dataset.csv")

df.head()

,Order_ID,Customer_ID,Customer_Name,City,Order_Date,Product_Category,Quantity,Unit_Price,Discount,Payment_Method,Order_Status,Customer_Rating,Delivery_Days,Total_Amount
0,ORD10001,CUST1001,Customer_1,NaN,01-01-2025,Sports,5,11202.0,15,Net Banking,Returned,1.0,6.0,47608.5
1,ORD10002,CUST1002,Customer_2,Ghaziabad,01-01-2025,Electronics,3,9718.0,20,Cash on Delivery,Shipped,4.0,6.0,23323.2
2,ORD10003,CUST1003,Customer_3,Faridabad,02-01-2025,Electronics,5,13458.0,0,Net Banking,Delivered,4.0,5.0,67290.0
3,ORD10004,CUST1004,Customer_4,Lucknow,02-01-2025,Electronics,2,1219.0,0,Debit Card,Returned,2.0,3.0,2438.0
4,ORD10005,CUST1005,Customer_5,Gurgaon,03-01-2025,Grocery,1,5437.0,0,Cash on Delivery,Delivered,2.0,6.0,5437.0


In [2]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

df.info()

Rows: 1010
Columns: 14
<class 'pandas.DataFrame'>
RangeIndex: 1010 entries, 0 to 1009
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Order_ID          1010 non-null   str    
 1   Customer_ID       1010 non-null   str    
 2   Customer_Name     1010 non-null   str    
 3   City              980 non-null    str    
 4   Order_Date        1010 non-null   str    
 5   Product_Category  1010 non-null   str    
 6   Quantity          1010 non-null   int64  
 7   Unit_Price        989 non-null    str    
 8   Discount          1010 non-null   int64  
 9   Payment_Method    985 non-null    str    
 10  Order_Status      1010 non-null   str    
 11  Customer_Rating   959 non-null    float64
 12  Delivery_Days     978 non-null    float64
 13  Total_Amount      1010 non-null   float64
dtypes: float64(3), int64(2), str(9)
memory usage: 110.6 KB


In [3]:
print("Duplicate rows:", df.duplicated().sum())

print("\nMissing values:")
print(df.isnull().sum())

Duplicate rows: 8

Missing values:
Order_ID             0
Customer_ID          0
Customer_Name        0
City                30
Order_Date           0
Product_Category     0
Quantity             0
Unit_Price          21
Discount             0
Payment_Method      25
Order_Status         0
Customer_Rating     51
Delivery_Days       32
Total_Amount         0
dtype: int64


In [4]:
df = df.drop_duplicates()

print("Duplicate rows after cleaning:", df.duplicated().sum())
print("Rows after removing duplicates:", len(df))

Duplicate rows after cleaning: 0
Rows after removing duplicates: 1002


In [6]:
text_columns = [
    "Customer_Name",
    "City",
    "Product_Category",
    "Payment_Method",
    "Order_Status"
]

for col in text_columns:
    df[col] = df[col].str.strip()

In [7]:
df["City"] = df["City"].replace({
    "delhi": "Delhi",
    "NOIDA": "Noida"
})

df["Product_Category"] = df["Product_Category"].str.title()

df["Payment_Method"] = df["Payment_Method"].replace({
    "upi": "UPI",
    "Credit card": "Credit Card"
})

df["Order_Status"] = df["Order_Status"].str.title()

In [8]:
df["Unit_Price"] = (
    df["Unit_Price"]
    .astype(str)
    .str.replace("₹", "", regex=False)
    .str.replace("INR", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)

df["Unit_Price"] = pd.to_numeric(
    df["Unit_Price"],
    errors="coerce"
)

df["Unit_Price"].head()

0    11202.0
1     9718.0
2    13458.0
3     1219.0
4     5437.0
Name: Unit_Price, dtype: float64

In [9]:
df.loc[df["Quantity"] <= 0, "Quantity"] = np.nan

df.loc[
    ~df["Customer_Rating"].between(1, 5),
    "Customer_Rating"
] = np.nan

df.loc[df["Delivery_Days"] <= 0, "Delivery_Days"] = np.nan

df.loc[df["Unit_Price"] <= 0, "Unit_Price"] = np.nan

In [10]:
df["City"] = df["City"].fillna(
    df["City"].mode()[0]
)

df["Payment_Method"] = df["Payment_Method"].fillna(
    df["Payment_Method"].mode()[0]
)

numeric_columns = [
    "Unit_Price",
    "Quantity",
    "Customer_Rating",
    "Delivery_Days"
]

for col in numeric_columns:
    df[col] = df[col].fillna(
        df[col].median()
    )

In [11]:
df["Order_Date"] = pd.to_datetime(
    df["Order_Date"],
    errors="coerce",
    dayfirst=True
)

print("Invalid dates:", df["Order_Date"].isnull().sum())

Invalid dates: 30


In [12]:
df = df.dropna(subset=["Order_Date"])

print("Rows after removing invalid dates:", len(df))

Rows after removing invalid dates: 972


In [13]:
df["Order_Date"] = df["Order_Date"].dt.strftime("%Y-%m-%d")

df["Order_Date"].head()

0    2025-01-01
1    2025-01-01
2    2025-01-02
3    2025-01-02
4    2025-01-03
Name: Order_Date, dtype: str

In [14]:
df["Total_Amount"] = (
    df["Quantity"]
    * df["Unit_Price"]
    * (1 - df["Discount"] / 100)
).round(2)

df.head()

,Order_ID,Customer_ID,Customer_Name,City,Order_Date,Product_Category,Quantity,Unit_Price,Discount,Payment_Method,Order_Status,Customer_Rating,Delivery_Days,Total_Amount
0,ORD10001,CUST1001,Customer_1,Ghaziabad,2025-01-01,Sports,5.0,11202.0,15,Net Banking,Returned,1.0,6.0,47608.5
1,ORD10002,CUST1002,Customer_2,Ghaziabad,2025-01-01,Electronics,3.0,9718.0,20,Cash on Delivery,Shipped,4.0,6.0,23323.2
2,ORD10003,CUST1003,Customer_3,Faridabad,2025-01-02,Electronics,5.0,13458.0,0,Net Banking,Delivered,4.0,5.0,67290.0
3,ORD10004,CUST1004,Customer_4,Lucknow,2025-01-02,Electronics,2.0,1219.0,0,Debit Card,Returned,2.0,3.0,2438.0
4,ORD10005,CUST1005,Customer_5,Gurgaon,2025-01-03,Grocery,1.0,5437.0,0,Cash on Delivery,Delivered,2.0,6.0,5437.0


In [15]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

Rows: 972
Columns: 14

Missing values:
Order_ID            0
Customer_ID         0
Customer_Name       0
City                0
Order_Date          0
Product_Category    0
Quantity            0
Unit_Price          0
Discount            0
Payment_Method      0
Order_Status        0
Customer_Rating     0
Delivery_Days       0
Total_Amount        0
dtype: int64

Duplicate rows:
0


In [16]:
df.to_csv(
    "cleaned_ecommerce_dataset.csv",
    index=False
)

In [17]:
comparison = pd.DataFrame({
    "Metric": [
        "Rows",
        "Duplicate Rows",
        "Missing Values"
    ],
    "Before": [
        1010,
        8,
        159
    ],
    "After": [
        len(df),
        df.duplicated().sum(),
        df.isnull().sum().sum()
    ]
})

comparison

,Metric,Before,After
0,Rows,1010,972
1,Duplicate Rows,8,0
2,Missing Values,159,0


In [18]:
df.head(10)

,Order_ID,Customer_ID,Customer_Name,City,Order_Date,Product_Category,Quantity,Unit_Price,Discount,Payment_Method,Order_Status,Customer_Rating,Delivery_Days,Total_Amount
0,ORD10001,CUST1001,Customer_1,Ghaziabad,2025-01-01,Sports,5.0,11202.0,15,Net Banking,Returned,1.0,6.0,47608.5
1,ORD10002,CUST1002,Customer_2,Ghaziabad,2025-01-01,Electronics,3.0,9718.0,20,Cash on Delivery,Shipped,4.0,6.0,23323.2
2,ORD10003,CUST1003,Customer_3,Faridabad,2025-01-02,Electronics,5.0,13458.0,0,Net Banking,Delivered,4.0,5.0,67290.0
3,ORD10004,CUST1004,Customer_4,Lucknow,2025-01-02,Electronics,2.0,1219.0,0,Debit Card,Returned,2.0,3.0,2438.0
4,ORD10005,CUST1005,Customer_5,Gurgaon,2025-01-03,Grocery,1.0,5437.0,0,Cash on Delivery,Delivered,2.0,6.0,5437.0
5,ORD10006,CUST1006,Customer_6,Delhi,2025-01-03,Electronics,3.0,10228.0,0,UPI,Delivered,2.0,1.0,30684.0
6,ORD10007,CUST1007,Customer_7,Delhi,2025-01-04,Electronics,2.0,12648.0,10,UPI,Delivered,3.0,4.0,22766.4
7,ORD10008,CUST1008,Customer_8,Noida,2025-01-04,Electronics,2.0,10840.0,25,Credit Card,Delivered,2.0,8.0,16260.0
8,ORD10009,CUST1009,Customer_9,Noida,2025-01-05,Grocery,3.0,12685.0,10,Credit Card,Returned,4.0,4.0,34249.5
9,ORD10010,CUST1010,Customer_10,Noida,2025-01-05,Grocery,5.0,1483.0,10,UPI,Cancelled,3.0,3.0,6673.5


In [20]:
comparison

,Metric,Before,After
0,Rows,1010,972
1,Duplicate Rows,8,0
2,Missing Values,159,0
